# Object-Relational Mapping (ORM)
-- это техника программирования, которая позволяет работать с данными в реляционной базе данных (SQL), используя ООП

- Еще более высокий уровень абстракции
- таблица - класс python



> **Например:** Запрос SELECT * FROM users возвращает таблицу. ORM преобразует каждую строку в объект User, где user.name обращается к столбцу name



### Пример ORM - SQLAlchemy

<img src="https://avatars.mds.yandex.net/get-pdb/2835522/84c43db6-4db7-44ab-8fd5-442ee4ecf2a2/s1200" style="height: 600px">

**SQLAlchemy состоит из двух частей**
1. SQLAlchemy Core ("Абстракция над SQL")


> **Суть:** Вы работаете с таблицами, столбцами и соединениями, но пишете их на Python, а не строками SQL

2. SQLAlchemy ORM ("Слой маппинга")


> **Суть:** Вы объявляете классы Python (class User), а ORM сам решает, как превратить их в строки таблицы





❗  Актуальная версия `SQLAlchemy 2.0`

# Преамбула

In [1]:
!rm chinook.db
!unzip chinook.zip

rm: cannot remove 'chinook.db': No such file or directory
Archive:  chinook.zip
  inflating: chinook.db              


#Core - engine



```
sqlalchemy.create_engine(url, echo=False, **kwargs)
```
>Создаёт и возвращает экземпляр Engine — основной точку входа для взаимодействия SQLAlchemy с базой данных. Engine управляет пулом соединений, диалектом (SQL диалект конкретной СУБД) и стратегиями выполнения запросов

- `url` : str | URL — строка подключения или объект URL
- `echo` : bool — Необязательный, по умолчанию `False`. Если True, все SQL-запросы выводятся в stdout.
- `**kwargs` : dict — Необязательный. Дополнительные параметры (pool_size, max_overflow, pool_recycle, connect_args и др.)







In [ ]:
from sqlalchemy import create_engine

In [31]:
engine = create_engine('sqlite+pysqlite:///chinook.db', echo=False)

# Модель

In [4]:
from sqlalchemy.orm import DeclarativeBase

In [14]:
class Base(DeclarativeBase): # базовая модель, от нее будем наследовать наши классы
	pass

❗  Почему это важно?

In [10]:
class User(DeclarativeBase):  # Прямое наследование от DeclarativeBase
    __tablename__ = "users"

InvalidRequestError: Cannot use 'DeclarativeBase' directly as a declarative base class. Create a Base by creating a subclass of it.

## Column




```
Column(name, type_, *args, **kwargs)
```


> Определяет столбец в таблице базы данных. Используется внутри класса модели для сопоставления атрибута класса с колонкой в таблице

- `name` : str — Имя столбца в БД. Если не указан, используется имя атрибута класса.
- `type_` : TypeEngine — Обязательный. Тип данных столбца (Integer, String, DateTime, Boolean и др.)
- `*args` : Any — Дополнительные ограничения (ForeignKey, CheckConstraint, PrimaryKeyConstraint и др.)
- `**kwargs` : dict — Дополнительные параметры (primary_key, nullable, default, unique, index и др.)

In [12]:
from sqlalchemy import Column, Integer, String, Float

In [15]:
# Определяем модель Track, которая будет сопоставлена с таблицей "tracks"
class Track(Base):
    __tablename__ = 'tracks'  # атрибут, содержащий имя моделируемой таблицы

    # список атрибутов
    track_id = Column(Integer, name='TrackId', primary_key=True)    # хотя бы один атрибут
                                                                    # должен являться первичным ключом
    Name = Column(String)
    GenreId = Column(Integer)
    UnitPrice = Column(Float)
    MediaTypeId = Column(Integer)
    Milliseconds = Column(Integer)

In [32]:
# создаем таблицы
Base.metadata.create_all(bind=engine)

#Сессия



```
Session(bind=None, autoflush=True, **kwargs)
```
> Фабрика (класс-построитель), которая создаёт новые объекты Session. Session — это рабочая единица, через которую идёт всё взаимодействие с БД: добавление, удаление, изменение и запрос объектов

- `bind` : Engine | Connection — Объект Engine (результат create_engine) или Connection. Привязывает сессию к конкретной БД
- `autoflush` : bool — Если True (по умолчанию), перед каждым запросом автоматически отправляет изменения в БД (flush), но без коммита



In [19]:
from sqlalchemy.orm import Session

In [20]:
# создаем саму сессию базы данных
with Session(autoflush=False, bind=engine) as session:
    pass

### Операции

In [33]:
from sqlalchemy import select

with Session(autoflush=False, bind=engine) as session:
  res = session.query(Track).all() # получение всех объектов
  for item in res[:5]:
        print(f"{item.track_id}, {item.Name}")

  print()
  # получение одного объекта по id
  first = session.get(Track, 1)
  print(f"Результат: {first.Name}")

1, For Those About To Rock (We Salute You)
2, Balls to the Wall
3, Fast As a Shark
4, Restless and Wild
5, Princess of the Dawn

Результат: For Those About To Rock (We Salute You)


### Фильтрация



```
Query.filter(*criterion)
```
> Применяет фильтрацию к запросу, возвращая новый Query объект с дополнительными условиями

- `*criterion` : SQLExpression — Одно или несколько условий фильтрации. Несколько переданных условий объединяются через AND

```
- `==` / `!=`           — Равенство / Неравенство
- `>` / `>=`            — Больше / Больше или равно
- `<` / `<=`            — Меньше / Меньше или равно
- `&`                   — Логическое И (AND)
- `|`                   — Логическое ИЛИ (OR)
- `~`                   — Логическое НЕ (NOT)
- `.in_()`              — Проверка вхождения в список
- `.like()` / `.ilike()`— Поиск по шаблону (регистрозависимый / нечувствительный)
- `.between()`          — Проверка попадания в диапазон
- `.is_()` / `.is_not()`— Проверка на NULL
- `.startswith()`       — Начинается с подстроки
- `.endswith()`         — Заканчивается на подстроку
- `.contains()`         — Содержит подстроку
```

In [34]:
with Session(autoflush=False, bind=engine) as session:
  res = session.query(Track).filter(Track.Milliseconds > 400000)
  for item in res[:5]:
        print(f"{item.track_id}, {item.Name}, {item.Milliseconds}")

50, You Oughta Know (Alternate), 491885
78, Master Of Puppets, 436453
124, Snoopy's search-Red baron, 456071
127, Stratus, 582086
142, No More Tears, 555075


## Обновление

In [ ]:
session.commit()

## Удаление

In [37]:
with Session(autoflush=False, bind=engine) as session:
  res = session.query(Track).filter(Track.Milliseconds > 400000).first() # получаем первый объект фильтрации
  print(f"{res.track_id}, {res.Name}, {res.Milliseconds}")
  session.delete(res)  # удаляем объект
  session.commit()     # сохраняем изменения

50, You Oughta Know (Alternate), 491885


Проверим, что удалился

In [38]:
with Session(autoflush=False, bind=engine) as session:
  res = session.query(Track).filter(Track.Milliseconds > 400000).first() # получаем первый объект фильтрации
  print(f"{res.track_id}, {res.Name}, {res.Milliseconds}")

78, Master Of Puppets, 436453


## Добавление строк



```
Session.add(instance)
```

> Добавляет объект в сессию. Объект становится отслеживаемым (pending) и будет вставлен в БД при следующем вызове commit() или flush()
- `instance` : object — Экземпляр класса модели (объект ORM), который нужно добавить в сессию.



In [42]:
# создаем сессию подключения к бд
with Session(autoflush=False, bind=engine) as session:
    track = Track(track_id="20220", Name="My song", GenreId=1, UnitPrice=10, MediaTypeId=1,Milliseconds=10000)
    session.add(track)     # добавляем в бд
    session.commit()     # сохраняем изменения
    print(track.track_id)   # можно получить установленный id

20220


`Session.rollback()`


> Откатывает текущую транзакцию. Все изменения, сделанные в сессии с момента последнего commit() или после предыдущего rollback(), отменяются. Все объекты, добавленные или изменённые, возвращаются в состояние, которое было до начала транзакции



In [58]:
# создаем сессию подключения к бд
with Session(autoflush=True, bind=engine) as session: # autoflush=True. Отправляем в БД, НО не фиксируем (commit бы фиксировал)
    track = Track(track_id="20225", Name="My song", GenreId=1, UnitPrice=10, MediaTypeId=1,Milliseconds=10000)
    session.add(track)     # добавляем в бд
    print(session.get(Track, 20225).Name)
    session.rollback()
    print(session.get(Track, 20225).Name)

My song


AttributeError: 'NoneType' object has no attribute 'Name'